# ESM-DMS TpoR local small-model analysis

This is a local-only version of `real_data_esm_analysis.ipynb` for the TpoR cellular dataset. It runs the `esmDMS` class methods directly in the notebook, uses repo-local data paths, and uses the smaller `esm2_t6_8M_UR50D` ESM-2 model.

No Slurm job creation, submission, polling, or scratch-copy workflow is included here.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

# Keep this checkout ahead of the older sibling esmDMS path inserted by
# embedding_scripts.embed_sequences during import.
sys.path.insert(0, str(REPO_ROOT))
import popDMS  # noqa: F401 - ensure esmDMS resolves the repo-local module
from esmDMS import CellularDMSInput, ESMDMSConfig, esmDMS
if sys.path[0] != str(REPO_ROOT):
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw_data"
ANALYSIS_DIR = DATA_DIR / "esm_data_analysis_local_small"
SEQUENCE_DIR = ANALYSIS_DIR / "sequence_data"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

for path in (SEQUENCE_DIR, FIGURE_DIR, TABLE_DIR):
    path.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
REPO_ROOT


## Dataset

TpoR uses the local MaveDB-style nucleotide counts and the local nucleotide reference sequence under `data/raw_data`.


In [ ]:
DATASET = "TpoR"
DATASET_KIND = {DATASET: "cellular"}

DATASETS = {
    DATASET: CellularDMSInput(
        reference_nuc_path=RAW_DIR / "TpoR_reference_sequence.dat",
        mavedb_csv_path=RAW_DIR / "TpoR_nucleotide_counts.csv",
    )
}

pd.DataFrame(
    {
        "dataset": dataset,
        "kind": DATASET_KIND[dataset],
        "reference": str(input_data.reference_nuc_path),
        "counts": str(input_data.mavedb_csv_path),
        "save_dir": str(SEQUENCE_DIR / dataset),
    }
    for dataset, input_data in DATASETS.items()
)


## Local Controls

This notebook runs embeddings and inference in the notebook process. Disk caching is still enabled via `local_or_disk="both"`, so repeated local runs can reuse saved embeddings and inference results.


In [ ]:
EMBEDDING_MODEL = "esm2_t6_8M_UR50D"
EMBEDDING_TYPE = "mean_pool"

# esm2_t6 has hidden-state layers 0..6. Layer 0 is fastest and matches the
# existing local walkthrough cache layout; add more layers if needed.
LAYERS = [0]
REPRESENTATIVE_LAYER = 0

ABSTRACTION_METHOD = "Embeddings"
ABSTRACTION_PARAMS = {"norm_scheme": "per_feature"}
NORM_SCHEME = ABSTRACTION_PARAMS["norm_scheme"]

RUN_EMBEDDINGS = True
RUN_INFERENCE = True
PLOT_RESULTS = True


## Create Local Runner


In [ ]:
runners = {}

for dataset, input_data in DATASETS.items():
    config = ESMDMSConfig(
        embedding_model=EMBEDDING_MODEL,
        embedding_type=EMBEDDING_TYPE,
        local_or_disk="both",
        save_dir=str(SEQUENCE_DIR / dataset),
        dataset_name=dataset,
    )
    runners[dataset] = esmDMS(input_data=input_data, config=config)

runners


## Process Raw Data


In [ ]:
processing_rows = []

for dataset, runner in runners.items():
    runner.process_raw_data(drop_stop_codons=True)
    df = runner.sequence_dataframe
    processing_rows.append({
        "dataset": dataset,
        "kind": DATASET_KIND[dataset],
        "rows": len(df),
        "sequence_count": df["SequenceIndex"].nunique(),
        "replicate_count": df["Replicate"].nunique(),
        "generation_count": df["Generation"].nunique(),
    })

processing_summary = pd.DataFrame(processing_rows)
processing_summary.to_csv(TABLE_DIR / "TpoR_raw_processing_summary.csv", index=False)
processing_summary


## Local Embeddings

This calls `embed_all_sequences()` directly. The selected ESM model is small, but this cell can still take time because it embeds every unique TpoR protein sequence.


In [ ]:
if RUN_EMBEDDINGS:
    for dataset, runner in runners.items():
        for layer in LAYERS:
            runner.embed_all_sequences(layer=layer)

embedding_rows = []
for dataset, runner in runners.items():
    for layer in LAYERS:
        embedding_path = runner._embedding_path(layer, EMBEDDING_TYPE)
        base_embedding_path = runner._base_embedding_path(layer)
        embedding_rows.append({
            "dataset": dataset,
            "layer": layer,
            "base_embedding_path": str(base_embedding_path),
            "base_embedding_exists": base_embedding_path.exists(),
            "derived_embedding_path": str(embedding_path),
            "derived_embedding_exists": embedding_path.exists(),
        })

embedding_status = pd.DataFrame(embedding_rows)
embedding_status.to_csv(TABLE_DIR / "TpoR_local_embedding_status.csv", index=False)
embedding_status


## Local Inference

This calls `run_feature_inference()` directly and stores results in the local cache.


In [ ]:
inference_results = {}

if RUN_INFERENCE:
    for dataset, runner in runners.items():
        inference_results[dataset] = {}
        for layer in LAYERS:
            inference_results[dataset][layer] = runner.run_feature_inference(
                layer=layer,
                abstraction_method=ABSTRACTION_METHOD,
                abstraction_params=ABSTRACTION_PARAMS,
                embedding_type=EMBEDDING_TYPE,
            )
else:
    for dataset, runner in runners.items():
        inference_results[dataset] = {}
        for layer in LAYERS:
            inference_results[dataset][layer] = runner.load_inference_results(
                layer=layer,
                abstraction_method=ABSTRACTION_METHOD,
                norm_scheme=NORM_SCHEME,
                embedding_type=EMBEDDING_TYPE,
            )


## Inference Summary


In [ ]:
inference_rows = []

for dataset, layer_results in inference_results.items():
    for layer, result in layer_results.items():
        inference_rows.append({
            "dataset": dataset,
            "kind": DATASET_KIND[dataset],
            "model": EMBEDDING_MODEL,
            "embedding_type": EMBEDDING_TYPE,
            "layer": layer,
            "norm_scheme": NORM_SCHEME,
            "n_replicates": result.s.shape[0],
            "n_dimensions": result.s.shape[1],
            "gamma_opt": result.gamma_opt,
            "s_joint_mean": result.s_joint.mean(),
            "s_joint_std": result.s_joint.std(),
        })

inference_summary = pd.DataFrame(inference_rows)
inference_summary.to_csv(TABLE_DIR / "TpoR_local_inference_result_summary.csv", index=False)
inference_summary


## Replicate Consistency Plot


In [ ]:
if PLOT_RESULTS:
    for dataset, runner in runners.items():
        runner.plot_avg_rep_correlations_by_layer(
            layers=LAYERS,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            embedding_type=EMBEDDING_TYPE,
            comparison="selection",
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_{EMBEDDING_MODEL}_{EMBEDDING_TYPE}_selection_replicate_correlations_by_layer.png",
        )
        runner.plot_avg_rep_correlations_by_layer(
            layers=LAYERS,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            embedding_type=EMBEDDING_TYPE,
            comparison="fitness",
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_{EMBEDDING_MODEL}_{EMBEDDING_TYPE}_fitness_replicate_correlations_by_layer.png",
        )
        plt.close("all")


## Representative Replicate Scatter Plots


In [ ]:
if PLOT_RESULTS:
    for dataset, runner in runners.items():
        runner.plot_rep_sel_comps(
            layer=REPRESENTATIVE_LAYER,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            embedding_type=EMBEDDING_TYPE,
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_{EMBEDDING_MODEL}_{EMBEDDING_TYPE}_layer{REPRESENTATIVE_LAYER}_selection_replicate_scatter.png",
        )
        runner.plot_rep_fit_comps(
            layer=REPRESENTATIVE_LAYER,
            abstraction_method=ABSTRACTION_METHOD,
            norm_scheme=NORM_SCHEME,
            embedding_type=EMBEDDING_TYPE,
            label=dataset,
            output_path=FIGURE_DIR / f"{dataset}_{EMBEDDING_MODEL}_{EMBEDDING_TYPE}_layer{REPRESENTATIVE_LAYER}_fitness_replicate_scatter.png",
        )
        plt.close("all")


## Output Locations


In [ ]:
{
    "sequence_dir": SEQUENCE_DIR,
    "figure_dir": FIGURE_DIR,
    "table_dir": TABLE_DIR,
}
